## Preparation

In [1]:
# Test OPENAI connection:
# uv add requests minsearch openai jupyter python-dotenv
# This installs:
# requests - to fetch the FAQ dataset from the internet
# minsearch - a simple in-memory search engine for indexing and searching text
# openai - the OpenAI API client for calling the LLM
# jupyter - the notebook environment where we'll write and run code
# python-dotenv - to load API keys from a .env file

# Check that the OpenAI client works:

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
openai_client
# should be something like <openai.OpenAI at 0x18e6f7e02f0>

In [3]:
# define a function to talk to the LLM:

def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
# test it:

llm("Hey, what's up?")

"Hey! Not much—I'm here and ready to help. What’s going on?"

In [5]:
# if no prior context exist, then pure hallucination - like this course-specific question:

question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — in many cases, yes, you can join after a course has started.

A few things to check:
- **Enrollment rules:** Some courses allow late enrollment, others don’t.
- **Access to past material:** You may or may not get recordings, slides, or missed assignments.
- **Deadlines:** There could be a cutoff for joining and catching up.
- **Instructor approval:** Some courses require permission to join late.

If you want, I can help you draft a short message to the course instructor or organizer asking if it’s still possible to join.


In [6]:
# The LLM gives a generic answer. It might say "you can usually join" or "check the course website." 
# It doesn't know about our specific Zoomcamp courses, their enrollment policies, or their schedules. 
# It tries to be helpful, but has no idea about actual enrollment status or policies.

# This is different from a question like "how do I cook salmon?" - the LLM knows the answer because 
# cooking salmon is common knowledge. But our courses are not in the training data.

question = "how do I cook salmon?"
answer = llm(question)
print(answer)


Here are a few easy ways to cook salmon:

### 1) Pan-seared salmon
**Best for:** crispy skin, fast cooking  
1. Pat the salmon dry and season with salt and pepper.  
2. Heat a skillet over medium-high heat with a little oil.  
3. Place salmon skin-side down and cook 4–6 minutes.  
4. Flip and cook 1–3 minutes more, depending on thickness.  
5. It’s done when it flakes easily and is opaque in the center.

### 2) Baked salmon
**Best for:** simple, hands-off cooking  
1. Preheat oven to **400°F (200°C)**.  
2. Put salmon on a lined baking sheet or in a baking dish.  
3. Season with salt, pepper, olive oil, lemon, garlic, herbs, etc.  
4. Bake for **12–15 minutes** for a typical fillet, or until it flakes easily.

### 3) Grilled salmon
**Best for:** smoky flavor  
1. Preheat grill to medium-high.  
2. Oil the grill grates and the salmon.  
3. Grill skin-side down for about **6–8 minutes**.  
4. Flip carefully and cook **2–4 minutes** more.

### Easy flavor ideas
- Lemon + garlic + butter
-

## RAG - adding context and loading docs

In [7]:
# More context can fix this. The FAQ website has questions and answers about our courses.

# Copy some of that content into the prompt:

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


In [8]:
question = "I just discovered the course. Can I join now?"

# moving back to course

In [9]:
# Build a prompt that includes both the question and the context:

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""
prompt

'\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\nI just discovered the course. Can I join now?\n\nContext:\n\nI just discovered the course. Can I still join?\nYes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nCourse: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nYou don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nWhat is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?\nThe zoom link is only published to instructors/present

In [10]:
# Instead of sending the raw question to the LLM, we send this prompt:

answer = llm(prompt)
print(answer)

# Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.
# This is RAG - providing model a context to ground it down and reduce hallucinations


Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [11]:
# RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. 
# We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. 
# That search step is what gives the LLM the context it needs to answer correctly.

# What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. 
# What we want instead is to perform search automatically. We take the student's question, 
# find the most relevant documents, and send those to the LLM.

# In code, it looks like this:

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [12]:
# The LLM only sees the documents we hand it, so its answers are grounded in our data. 
# If the right document is retrieved, the answer is good. 
# If it's not, the LLM gets the wrong context and the answer is wrong. 
# Your model is only as good as your retrieval, so search quality matters a lot for RAG.

# The database and the LLM can be anything. 
# In this course we use minsearch and then sqlitesearch for search, and OpenAI for the LLM. 
# But you can swap any component for another and see what works better.

# The FAQ data is available as JSON from the DataTalks.Club website. 
# I maintain that site, so I made the data available at a JSON endpoint we can fetch directly.

# Let's fetch it:

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [13]:
courses_raw[:10]

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [14]:
# This returns a list of courses. Each course has a path field that points to its FAQ data.

# Let's fetch all the FAQ documents from all courses:

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    print(course_url)

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

https://datatalks.club/faq/json/data-engineering-zoomcamp.json
https://datatalks.club/faq/json/stock-markets-analytics-zoomcamp.json
https://datatalks.club/faq/json/ai-dev-tools-zoomcamp.json
https://datatalks.club/faq/json/llm-zoomcamp.json
https://datatalks.club/faq/json/mlops-zoomcamp.json
https://datatalks.club/faq/json/machine-learning-zoomcamp.json


1342

In [15]:
# Each entry has:

# id - unique identifier
# course - course slug (e.g., machine-learning-zoomcamp)
# section - which section of the course
# question - the FAQ question
# answer - the FAQ answer
# Let's look at one:

documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [16]:
# In the RAG pipeline, this dataset is our knowledge base:

# We index all the documents (the search step)
# When a student asks a question, we search the index
# The search returns the most relevant FAQ entries
# We give those entries to the LLM as context
# The LLM generates an answer based on the context
# The question and answer fields contain the text we'll search through. 
# The course field lets us filter by course. For example, if a student asks 
# about the data engineering course, we skip results from the ML course. 
# The section field helps with ranking - knowing which part of the course a question belongs to is useful context.



## Search in general and minsearch

In [18]:
# Search basics
# At its core, every search engine does the same thing. It takes a query, scores every document for similarity, and returns the top results.

# Formally, there is a similarity function:

# score = sim(query, document)

# For each document in the database, you compute this score. Then you rank all documents by score and return the top N. 
# What makes a search engine different from another search engine is what sim actually computes.

# text/lexical search (covered in this section): sim counts how many words the query and the document share. 
# It looks at the surface form, the actual words, and matches them exactly.

# vector/semantic search (covered in module 2): sim compares the meaning of the query and the document. 
# Same function, different similarity measure.

# Consider these two questions:

# "Can I still join the course after the start date?"
# "Is it possible to enroll late?"
# They mean the same thing, but share almost no keywords. "Join" is not "enroll", "course" is absent, 
# "start date" is not "late". A text search engine would struggle to match them, because it only sees words.

# We'll see how vector search solves this later. For now, let's build text search with minsearch.

In [19]:
# Indexing with minsearch
# We already have the documents list from the previous section. Now let's index it.

# Searching matters because we have around 1100 documents. Sending all of them to the LLM would be expensive and slow. 
# The model would get confused with too much data. Search finds the most relevant documents to send instead.

# There are many search libraries you can use - Apache Lucene, Elasticsearch, Solr, and others. But these are somewhat heavy. 
# For example, to run Elasticsearch, you need to start a Docker container.

# minsearch is a simple in-memory search engine. It's lightweight, so it runs anywhere Python runs,
# including Google Colab where you can't start a Docker container. It's a toy implementation, not production ready, 
# but it illustrates how search engines work and it gives good results.

# I should add a disclaimer: I wrote this library and I maintain it. It started as a single Python file in the first edition 
# of LLM Zoomcamp. I wanted to show that keyword search isn't magic. We wrote it together as part of the Build a Search Engine 
# workshop (see the code).

# It turned out useful beyond teaching. After two years across many projects, it's pretty reliable for small datasets.

# The concepts in minsearch are the same as in Elasticsearch (which comes from Lucene): text fields, keyword fields, boosting, 
# filtering. I borrowed those terms from Elasticsearch on purpose, since I wanted a lightweight stand-in for it. 
# So what you learn here transfers directly.

# We'll index the question, section, and answer fields as text (they'll be tokenized and ranked), 
# and the course field as a keyword (for filtering).

# The index tokenizes text fields and treats keyword fields as exact strings.

# Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. 
# It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. 
# So question, section, and answer are text fields.

# Keyword fields are for exact matching. Think of a SQL query like SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'. 
# The results must come from the specified course, no matter what ranking or boosting you do for text fields.

# You use keyword fields to restrict the search space to a particular subset. In our case, we have four courses. 
# Say you're taking the LLM course and ask a question. You don't want answers from the MLOps or machine learning courses mixed in.

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)
index

# That's it, the index is built. The fit name comes from scikit-learn, where you fit a model on data. 
# Here we fit an index on our documents.
# takes 1 min or so on my laptop - <minsearch.minsearch.Index at 0x1841ff46e40>

In [20]:
# Let's try a search with the question we used before:

question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [21]:
# We get back 5 results from the LLM Zoomcamp FAQ. The best match is the FAQ entry "I just discovered the course. 
# Can I still join?" which is exactly what we need.

# Here are all the questions:

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']